In [ ]:
import os
import re
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

with open("genome_id.txt", "r", encoding="utf-8", errors="ignore") as f:
    raw = f.read()

# Extract IDs or fallback to cleaned lines
ids = list(dict.fromkeys(re.findall(r"\b\d+\.\d+\b", raw)))
if not ids:
    ids = [
        line.strip().strip(",'\"[]")
        for line in raw.splitlines()
        if line.strip().strip(",'\"[]")
    ]

os.makedirs("bvbrc_fna", exist_ok=True)
failed = []

def fetch(gid):
    out = os.path.join("bvbrc_fna", f"{gid}.fna")
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return ("SKIP", gid)
    url = f"https://www.bv-brc.org/api/genome_sequence/?eq(genome_id,{gid})&http_accept=application/dna+fasta"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=25) as resp:
            data = resp.read()
        if b">" in data:
            with open(out, "wb") as fh:
                fh.write(data)
            return ("OK", gid)
        return ("FAIL", gid)
    except Exception:
        return ("FAIL", gid)

total = len(ids)
print(f"Found {total} IDs")

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(fetch, gid) for gid in ids]
    for i, fut in enumerate(as_completed(futures), 1):
        status, gid = fut.result()
        print(f"[{i}/{total}] {status}: {gid}")
        if status == "FAIL":
            failed.append(gid)

if failed:
    with open("bvbrc_fna/failed.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(failed) + "\n")

saved = len([x for x in os.listdir("bvbrc_fna") if x.endswith(".fna")])
print(f"Done. Saved: {saved}, Failed: {len(failed)}")

Found 995 IDs
[1/995] OK: 562.96941
[2/995] OK: 562.142721
[3/995] OK: 562.193047
[4/995] OK: 562.36249
[5/995] OK: 562.9889
[6/995] OK: 562.61684
[7/995] OK: 562.22854
[8/995] OK: 562.99237
[9/995] OK: 562.47191
[10/995] OK: 562.98343
[11/995] OK: 562.22869
[12/995] OK: 562.174289
[13/995] OK: 562.23301
[14/995] OK: 562.192678
[15/995] FAIL: 562.6684
[16/995] OK: 562.94043
[17/995] OK: 562.38796
[18/995] OK: 562.72339
[19/995] OK: 562.22877
[20/995] OK: 562.67837
[21/995] OK: 562.175157
[22/995] OK: 562.46376
[23/995] OK: 562.90079
[24/995] OK: 562.79435
[25/995] OK: 562.36712
[26/995] OK: 562.46083
[27/995] OK: 562.172973
[28/995] OK: 562.9934
[29/995] OK: 562.65898
[30/995] OK: 562.100808
[31/995] OK: 562.70515
[32/995] OK: 562.97229
[33/995] OK: 562.47411
[34/995] OK: 562.91606
[35/995] OK: 562.29167
[36/995] OK: 244319.31
[37/995] OK: 562.185213
[38/995] OK: 562.141343
[39/995] OK: 562.175108
[40/995] OK: 562.30907
[41/995] FAIL: 562.18482
[42/995] OK: 562.144637
[43/995] OK: 562.